# ARC-AGI-3 — MyAgent

**Cómo funciona este notebook.** Kaggle lo ejecuta dos veces:

1. **Commit** (*Save & Run All*): sin servidor de juegos. Aquí se hace una prueba de humo
   de `MyAgent` dentro del framework real y se escribe un `submission.parquet` de relleno.
2. **Rerun de competencia** (al pulsar *Submit to Competition*): aparece el *gateway* con los
   juegos ocultos, se registra `MyAgent` en el framework y `main.py` lo juega todo.
   El gateway genera el `submission.parquet` real.

`agent.py`, `swarm.py`, `recorder.py` y `tracing.py` **ya vienen** en el dataset de la
competencia (`ARC-AGI-3-Agents/agents/`), así que no se copian aquí: solo se escribe
la subclase `MyAgent` (celda 2) y se enchufa al registro `AVAILABLE_AGENTS`.

Lo único que normalmente editas es la celda 2.


In [1]:
!pip install -q --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [2]:
%%writefile /tmp/my_agent.py
"""MyAgent — explorador por novedad para ARC-AGI-3.

Idea: cada frame se reduce a un hash. Para cada estado visto se guarda qué
acciones ya se probaron ahí, y siempre se elige una acción todavía no probada
en ese estado. Las acciones que no cambian nada quedan marcadas y no se repiten;
las que sí cambian el frame llevan a estados nuevos con opciones frescas.
ACTION6 (clic) apunta solo a celdas que no son fondo, no a coordenadas al azar.
"""
from __future__ import annotations

import hashlib
import random
from collections import defaultdict
from typing import Any

import numpy as np
from arcengine import FrameData, GameAction, GameState

from agents.agent import Agent


class MyAgent(Agent):
    MAX_ACTIONS = 400      # tope por juego; la perilla más importante
    CLICK_TARGETS = 16     # máximo de celdas candidatas para ACTION6 por estado

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.rng = random.Random(self.game_id)
        self.tried: dict[str, set] = defaultdict(set)

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def choose_action(
        self, frames: list[FrameData], latest_frame: FrameData
    ) -> GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            return GameAction.RESET

        grid = np.asarray(latest_frame.frame[-1], dtype=np.int16)
        state = hashlib.blake2b(grid.tobytes(), digest_size=8).hexdigest()

        legal = [GameAction.from_id(i) for i in latest_frame.available_actions if i]
        if not legal:
            legal = [a for a in GameAction if a is not GameAction.RESET]

        options = []
        for a in legal:
            if a.is_complex():
                options += [(a, xy) for xy in self._targets(grid, state)]
            else:
                options.append((a, None))

        fresh = [o for o in options if o not in self.tried[state]]
        action, xy = self.rng.choice(fresh or options)
        self.tried[state].add((action, xy))

        if xy is not None:
            action.set_data({"x": xy[0], "y": xy[1]})
        action.reasoning = {"state": state, "fresh": len(fresh)}
        return action

    def _targets(self, grid: np.ndarray, state: str) -> list[tuple[int, int]]:
        """Celdas distintas del color dominante (fondo), estables por estado."""
        if grid.ndim != 2 or grid.size == 0:
            return [(32, 32)]
        colors, counts = np.unique(grid, return_counts=True)
        rows, cols = np.nonzero(grid != colors[counts.argmax()])
        cells = [(int(c), int(r)) for r, c in zip(rows, cols)]   # (x, y)
        if not cells:
            return [(32, 32)]
        k = min(self.CLICK_TARGETS, len(cells))
        return random.Random(state).sample(cells, k)


Writing /tmp/my_agent.py


In [3]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # ── Prueba de humo: MyAgent dentro del Agent.main() real, contra un juego de juguete ──
    import shutil, sys, importlib.util
    import numpy as np
    import pandas as pd
    from arcengine import FrameDataRaw, GameAction, GameState

    shutil.copytree('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents/agents', '/tmp/fw/agents', dirs_exist_ok=True)
    with open('/tmp/fw/agents/__init__.py', 'w') as f:   # sin langgraph/smolagents
        f.write('from .agent import Agent, Playback\nfrom .swarm import Swarm\nAVAILABLE_AGENTS = {}\n')
    sys.path.insert(0, '/tmp/fw')

    spec = importlib.util.spec_from_file_location('my_agent', '/tmp/my_agent.py')
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

    class ToyEnv:
        """Jugador (3) se mueve con ACTION1-4; clic sobre el objetivo (5) gana."""
        def __init__(self): self.s, self.p = GameState.NOT_PLAYED, [10, 10]
        def _raw(self):
            g = np.zeros((64, 64), np.int8); g[40, 50] = 5; g[self.p[1], self.p[0]] = 3
            r = FrameDataRaw(game_id='toy', state=self.s, available_actions=[1, 2, 3, 4, 6], guid='toy')
            r.frame = [g]; return r
        @property
        def observation_space(self): return self._raw()
        def step(self, a, data=None, reasoning=None):
            if a is GameAction.RESET: self.s, self.p = GameState.NOT_FINISHED, [10, 10]
            elif a is GameAction.ACTION6:
                if (data['x'], data['y']) == (50, 40): self.s = GameState.WIN
            else:
                dx, dy = {1: (0, -1), 2: (0, 1), 3: (-1, 0), 4: (1, 0)}[a.value]
                self.p = [min(63, max(0, self.p[0] + dx)), min(63, max(0, self.p[1] + dy))]
            return self._raw()

    agent = mod.MyAgent(card_id='smoke', game_id='toy', agent_name='myagent',
                        ROOT_URL='http://localhost', record=False, arc_env=ToyEnv())
    agent.main()
    print(f'Prueba de humo OK → estado={agent.frames[-1].state.name}, acciones={agent.action_counter}')

    # ── submission.parquet de relleno para que el commit sea aceptado ──
    pd.DataFrame([['1_0', '1', True, 1]],
                 columns=['row_id', 'game_id', 'end_of_game', 'score']
                 ).to_parquet('/kaggle/working/submission.parquet', index=False)


Prueba de humo OK → estado=WIN, acciones=10


In [4]:
import os
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Esperar a que el gateway (sidecar con los juegos ocultos) esté listo.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copiar el framework oficial a un lugar escribible y meter MyAgent como template.
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents
    !cp /tmp/my_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Registro de agentes: esto es lo que Swarm lee en `from . import AVAILABLE_AGENTS`.
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {'myagent': MyAgent}
""")

    # Apuntar el framework al gateway.
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Swarm: un hilo por juego. El gateway registra cada acción y emite submission.parquet.
    !cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg python main.py --agent myagent
